In [1]:
"""
======================================================================
 PHASE 1: CACHING FUNDAMENTALS  —  Runnable Playground
======================================================================
 Every Phase 1 concept lives here as working code.
 Nothing to memorize — just run it and read the output.

 Each PART teaches one concept. Read top to bottom.
======================================================================
"""

'\n======================================================================\n PHASE 1: CACHING FUNDAMENTALS  —  Runnable Playground\n======================================================================\n Every Phase 1 concept lives here as working code.\n Nothing to memorize — just run it and read the output.\n\n Each PART teaches one concept. Read top to bottom.\n======================================================================\n'

In [2]:
import time
from collections import OrderedDict, Counter, deque

In [3]:
# ─────────────────────────────────────────────────────────────────────
# PART 0:  A fake "slow database"
# ─────────────────────────────────────────────────────────────────────
# Real databases are slow (disk + network). We fake that with a 0.4s delay.
# This delay is the whole reason cache exists.

def slow_database_query(user_id):
    time.sleep(0.4)
    return {"id": user_id, "name": f"User_{user_id}"}

In [4]:
# ─────────────────────────────────────────────────────────────────────
# PART 1:  The simplest cache (a dict)  —  HIT vs MISS
# ─────────────────────────────────────────────────────────────────────
# This is the "Cache-Aside" pattern: look in cache, on miss hit the DB,
# then store the result back in cache.

cache = {}

def get_user(user_id):
    if user_id in cache:
        print(f"   CACHE HIT   -> user {user_id} (fast, never touched the DB)")
        return cache[user_id]

    print(f"   CACHE MISS  -> user {user_id} (going to the DB, slow...)")
    data = slow_database_query(user_id)
    cache[user_id] = data
    return data


def demo_part1():
    print("\n=== PART 1: Hit vs Miss (watch the speed difference) ===")
    t = time.time(); get_user(1); print(f"      took: {time.time()-t:.2f}s")  # MISS
    t = time.time(); get_user(1); print(f"      took: {time.time()-t:.2f}s")  # HIT
    t = time.time(); get_user(2); print(f"      took: {time.time()-t:.2f}s")  # MISS

In [5]:
# ─────────────────────────────────────────────────────────────────────
# PART 2:  TTL + EXPIRATION  —  data can't live in cache forever
# ─────────────────────────────────────────────────────────────────────
# Store an expiry time with each key. Once the time passes, it's invalid.

class TTLCache:
    def __init__(self):
        self.store = {}                   # key -> (value, expire_at)

    def set(self, key, value, ttl_seconds):
        self.store[key] = (value, time.time() + ttl_seconds)

    def get(self, key):
        if key not in self.store:
            return None
        value, expire_at = self.store[key]
        if time.time() > expire_at:       # time is up = EXPIRED
            del self.store[key]           # lazy expiration: remove on access
            return None
        return value


def demo_part2():
    print("\n=== PART 2: TTL / Expiration ===")
    c = TTLCache()
    c.set("otp", "123456", ttl_seconds=1)          # OTP lives for 1 second
    print(f"   read right now:   {c.get('otp')}")   # valid
    time.sleep(1.1)
    print(f"   read after 1.1s:  {c.get('otp')}")   # None -> expired!

In [6]:
# ─────────────────────────────────────────────────────────────────────
# PART 3:  EVICTION POLICIES  —  cache is full, who gets kicked out?
# ─────────────────────────────────────────────────────────────────────

# LRU: Least Recently Used — evict the one you haven't used for the longest.
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.store = OrderedDict()        # remembers insertion/use order

    def get(self, key):
        if key not in self.store:
            return None
        self.store.move_to_end(key)       # just used -> now "most recent"
        return self.store[key]

    def set(self, key, value):
        if key in self.store:
            self.store.move_to_end(key)
        self.store[key] = value
        if len(self.store) > self.capacity:
            evicted, _ = self.store.popitem(last=False)   # least recently used
            print(f"      LRU evicted: {evicted}")


# LFU: Least Frequently Used — evict the one used the fewest times.
class LFUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.store = {}
        self.freq = Counter()             # how many times each key was used

    def get(self, key):
        if key not in self.store:
            return None
        self.freq[key] += 1
        return self.store[key]

    def set(self, key, value):
        if key not in self.store and len(self.store) >= self.capacity:
            lfu_key = min(self.freq, key=self.freq.get)   # least used
            del self.store[lfu_key]; del self.freq[lfu_key]
            print(f"      LFU evicted: {lfu_key}")
        self.store[key] = value
        self.freq[key] += 1


# FIFO: First In First Out — evict the oldest inserted (ignores usage).
class FIFOCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.store = {}
        self.order = deque()

    def get(self, key):
        return self.store.get(key)        # reading does NOT change order

    def set(self, key, value):
        if key not in self.store:
            self.order.append(key)
        self.store[key] = value
        if len(self.store) > self.capacity:
            oldest = self.order.popleft()
            del self.store[oldest]
            print(f"      FIFO evicted: {oldest}")


def demo_part3():
    print("\n=== PART 3: Eviction (capacity = 2) ===")

    print("   -- LRU --")
    lru = LRUCache(2)
    lru.set("A", 1); lru.set("B", 2)
    lru.get("A")                          # A just used -> B is now the old one
    lru.set("C", 3)                       # full -> B leaves (not recent)

    print("   -- LFU --")
    lfu = LFUCache(2)
    lfu.set("A", 1); lfu.set("B", 2)
    lfu.get("A"); lfu.get("A")            # used A twice
    lfu.set("C", 3)                       # B used least -> B leaves

    print("   -- FIFO --")
    fifo = FIFOCache(2)
    fifo.set("A", 1); fifo.set("B", 2)
    fifo.get("A")                         # using it doesn't help
    fifo.set("C", 3)                      # A was inserted first -> A leaves

In [7]:
# ─────────────────────────────────────────────────────────────────────
# PART 4:  WRITE PATTERNS  —  on update, how do cache & DB stay in sync?
# ─────────────────────────────────────────────────────────────────────

fake_db = {}                              # our fake database

# Write-Through: write to cache AND DB together (safe, a bit slow)
class WriteThroughCache:
    def __init__(self): self.cache = {}
    def write(self, key, value):
        self.cache[key] = value           # 1) cache
        fake_db[key] = value              # 2) DB — at the same time
    def read(self, key): return self.cache.get(key)

# Write-Back: write only to cache, flush to DB later (fast, risky)
class WriteBackCache:
    def __init__(self): self.cache = {}; self.dirty = set()
    def write(self, key, value):
        self.cache[key] = value
        self.dirty.add(key)               # DB doesn't know yet
    def flush(self):
        for k in self.dirty: fake_db[k] = self.cache[k]
        print(f"      flushed to DB: {sorted(self.dirty)}")
        self.dirty.clear()

# Write-Around: write straight to DB, bypass the cache
def write_around(key, value, cache_dict):
    fake_db[key] = value
    cache_dict.pop(key, None)             # drop stale copy from cache if any


def demo_part4():
    print("\n=== PART 4: Write Patterns ===")

    wt = WriteThroughCache()
    wt.write("x", 100)
    print(f"   Write-Through -> cache: {wt.read('x')} | DB: {fake_db.get('x')} (both at once)")

    wb = WriteBackCache()
    wb.write("y", 200)
    print(f"   Write-Back    -> cache: {wb.cache.get('y')} | DB: {fake_db.get('y')} (DB still None!)")
    wb.flush()
    print(f"                    after flush, DB: {fake_db.get('y')}")

    some_cache = {"z": "old"}
    write_around("z", 300, some_cache)
    print(f"   Write-Around  -> DB: {fake_db.get('z')} | cache: {some_cache.get('z')} (dropped from cache)")


# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    demo_part1()
    demo_part2()
    demo_part3()
    demo_part4()
    print("\nDone! Now tweak the values and run again — that's how it sticks.")


=== PART 1: Hit vs Miss (watch the speed difference) ===
   CACHE MISS  -> user 1 (going to the DB, slow...)
      took: 0.40s
   CACHE HIT   -> user 1 (fast, never touched the DB)
      took: 0.00s
   CACHE MISS  -> user 2 (going to the DB, slow...)
      took: 0.40s

=== PART 2: TTL / Expiration ===
   read right now:   123456
   read after 1.1s:  None

=== PART 3: Eviction (capacity = 2) ===
   -- LRU --
      LRU evicted: B
   -- LFU --
      LFU evicted: B
   -- FIFO --
      FIFO evicted: A

=== PART 4: Write Patterns ===
   Write-Through -> cache: 100 | DB: 100 (both at once)
   Write-Back    -> cache: 200 | DB: None (DB still None!)
      flushed to DB: ['y']
                    after flush, DB: 200
   Write-Around  -> DB: 300 | cache: None (dropped from cache)

Done! Now tweak the values and run again — that's how it sticks.
